# 02 — Calibração Científica do Limiar de Abstenção (`min_score`)
**SPEC-03 (SDD)** — Comparativo entre a linha de base da Engenharia (`TF-IDF min_score=0.12`, ADR-003) e o **Hybrid Retriever (Tabular DCB Gold + Vetorial/Léxico RRF)** sobre o `golden_set_v1.json`.

In [ ]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from sus_cidadao_ds.calibration.threshold_optimizer import ThresholdCalibrator
from sus_cidadao_ds.contracts.schemas import GoldenCase
from sus_cidadao_ds.core.config import ProjectPaths
from sus_cidadao_ds.retrieval.retrievers import HybridRetriever, TFIDFRetriever
from sus_cidadao_ds.sync.data_sync import DataSynchronizer

paths = ProjectPaths.from_root(ROOT)
syncer = DataSynchronizer(paths)
manifest = syncer.load_manifest()
meds = syncer.load_medications()
chunks = syncer.load_chunks()
golden = [GoldenCase.from_dict(x) for x in json.loads((paths.datasets_dir / "golden_set_v1.json").read_text())]

cal = ThresholdCalibrator().run_calibration(
    tfidf_retriever=TFIDFRetriever(chunks),
    hybrid_retriever=HybridRetriever(meds, chunks),
    golden_cases=golden,
    manifest=manifest,
)
print("Baseline TF-IDF (0.12):", cal.baseline_tfidf_012.to_dict())
print("Best TF-IDF:", cal.best_tfidf.to_dict())
print("Best Hybrid:", cal.best_hybrid.to_dict())